In [ ]:
import pandas as pd

# Input/output files
input_ehr_file = "EHR_cleaned.csv"
output_sample_file = "patient_data.csv"

# Read EHR dataset
ehr_df = pd.read_csv(input_ehr_file)

# Remove rows where apacheadmissiondx (diagnosis/symptoms) is missing or "unknown"
ehr_df = ehr_df.dropna(subset=['apacheadmissiondx'])
ehr_df = ehr_df[ehr_df['apacheadmissiondx'].str.lower() != "unknown"]

# Take 100 valid samples
sample_df = ehr_df.head(150).copy()

# Select required fields
patient_data = sample_df[['age', 'gender', 'apacheadmissiondx']].copy()
patient_data.rename(columns={'apacheadmissiondx': 'diagnosis'}, inplace=True)

# Use diagnosis text as synthetic "symptoms"
patient_data['symptoms'] = patient_data['diagnosis'].apply(
    lambda x: f"Symptoms related to {str(x).lower()}"
)

# Reorder
patient_data = patient_data[['age', 'gender', 'symptoms', 'diagnosis']]

# Save clean sample dataset
patient_data.to_csv(output_sample_file, index=False)
print(f"✅ Step 1 done → '{output_sample_file}' created with {len(patient_data)} valid rows (no unknown diagnosis)")
print(patient_data.head())



✅ Step 1 done → 'patient_data.csv' created with 150 valid rows (no unknown diagnosis)
  age gender                                           symptoms  \
0  45   male  Symptoms related to hypertension, uncontrolled...   
1  50   male  Symptoms related to ablation or mapping of car...   
2  83   male        Symptoms related to endarterectomy, carotid   
3  49   male  Symptoms related to infarction, acute myocardi...   
4  57   male  Symptoms related to cabg alone, coronary arter...   

                                           diagnosis  
0  hypertension, uncontrolled (for cerebrovascula...  
1  ablation or mapping of cardiac conduction pathway  
2                            endarterectomy, carotid  
3                  infarction, acute myocardial (mi)  
4        cabg alone, coronary artery bypass grafting  


In [ ]:
import pandas as pd

# --- Placeholder for Azure OpenAI API call ---
def generate_clinical_note_from_azure(age, gender, symptoms, diagnosis):
    """
    This function simulates generating a clinical note.
    Later, you can replace the inside with your actual Azure OpenAI API call.
    """
    return (
        f"Patient: {age}-year-old {gender.lower()}.\n"
        f"Presenting Complaints: {symptoms}.\n"
        f"Assessment: Likely {diagnosis.lower()} based on available information.\n"
        f"Plan: Recommend appropriate diagnostic tests (labs, imaging) and initiate treatment as clinically indicated."
    )

# --- Step 2: Generate Clinical Notes ---
input_file = "patient_data.csv"
output_file = "clinical_notes_output.csv"

# Read the cleaned patient dataset
try:
    patient_df = pd.read_csv(input_file)
except FileNotFoundError:
    print(f"❌ Error: '{input_file}' not found. Please run Step 1 first.")
    exit()

# Generate clinical notes
patient_df['clinical_note'] = patient_df.apply(
    lambda row: generate_clinical_note_from_azure(
        row['age'], row['gender'], row['symptoms'], row['diagnosis']
    ), axis=1
)

# Save the results
patient_df.to_csv(output_file, index=False)

print(f"✅ Step 2 complete → '{output_file}' created with {len(patient_df)} notes")
print(patient_df[['age', 'gender', 'symptoms', 'diagnosis', 'clinical_note']].head())


✅ Step 2 complete → 'clinical_notes_output.csv' created with 150 notes
  age gender                                           symptoms  \
0  45   male  Symptoms related to hypertension, uncontrolled...   
1  50   male  Symptoms related to ablation or mapping of car...   
2  83   male        Symptoms related to endarterectomy, carotid   
3  49   male  Symptoms related to infarction, acute myocardi...   
4  57   male  Symptoms related to cabg alone, coronary arter...   

                                           diagnosis  \
0  hypertension, uncontrolled (for cerebrovascula...   
1  ablation or mapping of cardiac conduction pathway   
2                            endarterectomy, carotid   
3                  infarction, acute myocardial (mi)   
4        cabg alone, coronary artery bypass grafting   

                                       clinical_note  
0  Patient: 45-year-old male.\nPresenting Complai...  
1  Patient: 50-year-old male.\nPresenting Complai...  
2  Patient: 83-year-old 